In [1]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import entropy
from scipy.special import kl_div, rel_entr
from scipy.spatial.distance import jensenshannon

import metrics_entropic as info

In [2]:
### data in

avila_emb = np.load('/Users/au582299/Repositories/open-ai-fun/avila_embeddings.npy')

def to_prob_dist(X):
    hist, _ = np.histogram(X, bins=5)
    hist = hist / hist.sum()
    return hist

avila_norm = [to_prob_dist(vector) for vector in avila_emb]
X = np.array(avila_norm)

## testing different KLD implementations

In [3]:
# sample
def knielbo_kld(p, q):
    """ KL-divergence for two probability distributions
    """
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)

    return np.sum(np.where(p != 0, (p-q) * np.log2(p / q), 0))

def my_kld_1(p, q):
    return p * np.log2(p / q) - p + q

def my_kld_2(p, q):
    return np.sum(p[i] * np.log2(p[i] / q[i]) for i in range(len(p)))

def simple_kld(p, q):
    return np.sum(p * np.log2(p / q))


In [7]:
# simple test

p = np.array([0.8, 0.2])
q = np.array([0.5, 0.5])

simple_comparison = {
    "knielbo_kld": knielbo_kld(p, q),
    "my_kld_1": my_kld_1(p, q).sum(),
    "my_kld_2": my_kld_2(p, q),
    "simple_kld": simple_kld(p, q),
    "scipy_kl_div": kl_div(p, q).sum(),
    "scipy_rel_entr": rel_entr(p, q).sum(),
    "scipy_entropy": entropy(p, q, base=2),
    "infodynamics_kld": info.kl_div(p, q),
}

simple_comparison

/var/folders/_1/x3y6hbdn5ljg_g6fr5kj88bs87mgrt/T/ipykernel_17861/4193400129.py:14: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  return np.sum(p[i] * np.log2(p[i] / q[i]) for i in range(len(p)))


{'knielbo_kld': 0.6,
 'my_kld_1': 0.2780719051126377,
 'my_kld_2': 0.27807190511263774,
 'simple_kld': 0.27807190511263774,
 'scipy_kl_div': 0.19274475702175747,
 'scipy_rel_entr': 0.19274475702175753,
 'scipy_entropy': 0.2780719051126378,
 'infodynamics_kld': 0.27807190511263774}

In [8]:
directionality_comparison = {
    "JSD(p | q)": jensenshannon(p, q, base=2),
    "JSD(q | p)": jensenshannon(q, p, base=2),
    "KLD(p | q)": entropy(p, q, base=2),
    "KLD(q | p)": entropy(q, p, base=2),
}

directionality_comparison

{'JSD(p | q)': 0.2703775285259666,
 'JSD(q | p)': 0.2703775285259666,
 'KLD(p | q)': 0.2780719051126378,
 'KLD(q | p)': 0.32192809488736235}

In [9]:
equivalence_comparison = {
    "scipy_kld": entropy(p, q, base=2),
    "my_kld": info.kl_div(p, q),
    "scipy_jsd": jensenshannon(p, q, base=2),
    "my_jsd": info.js_dist(p, q)
}

equivalence_comparison

{'scipy_kld': 0.2780719051126378,
 'my_kld': 0.27807190511263774,
 'scipy_jsd': 0.2703775285259666,
 'my_jsd': 0.2703775285259665}

In [13]:
p = np.array([0.8, 0.2])
q = np.array([[0.5, 0.5], [0.3, 0.7]])

vectorized_comparison = {
    "scipy_kld": entropy(p, q, base=2),
    "my_kld": info.kl_div(p, q),
    "scipy_jsd": jensenshannon(p, q, base=2),
    "my_jsd": info.js_dist(p, q)
}

vectorized_comparison

/Users/au582299/Repositories/open-ai-fun/.venv/lib/python3.11/site-packages/scipy/stats/_entropy.py:139: RuntimeWarning: invalid value encountered in divide
  qk = 1.0*qk / np.sum(qk, axis=axis, keepdims=True)
/Users/au582299/Repositories/open-ai-fun/.venv/lib/python3.11/site-packages/scipy/spatial/distance.py:1250: RuntimeWarning: invalid value encountered in divide
  q = q / np.sum(q, axis=axis, keepdims=True)


{'scipy_kld': array([ 1.15940257, -0.52064284,         nan]),
 'my_kld': 1.0486309201241921,
 'scipy_jsd': array([0.25395267, 0.31417829,        nan]),
 'my_jsd': 0.5498475118991435}

In [21]:
p0 = np.array([0.8, 0.2, 0.0])
q0 = np.array([[0.5, 0.5, 0.0], [0.3, 0.7, 0.0]])

def _rel_entr_vec(p, q):
    """
    Adapted from scipy.special.rel_entr
    """
    p = np.asarray(p) + np.finfo(float).eps
    q = np.asarray(q) + np.finfo(float).eps

    mask = (p > 0) & (q > 0)
    result = np.where(mask, p * np.log2(p / q), 0)
    return result

def kl_div_vec(p, q):
    return _rel_entr_vec(p, q).sum(axis=-1)

zero_comparison = {
    "no zeros": kl_div_vec(p, q),
    "yes zeros": kl_div_vec(p0, q0)
}

zero_comparison


{'no zeros': array([0.27807191, 0.77055902]),
 'yes zeros': array([0.27807191, 0.77055902])}